# Temperature-Gradient Sensitivity for the Objective Subtype PCA/Clustering

This notebook is a **separate sensitivity test** for the current `k = 3` objective subtype solution.

Main question:
- does the cluster story materially change if the `850 hPa |grad T|` clustering variable is removed?

Why this notebook exists:
- the `850 hPa` temperature-gradient composite still appears strongly terrain-linked even after display-only colorbar tightening
- we want to test whether that variable is actually needed in the subtype-definition math before masking terrain or rewriting Notebook 10

What this notebook does:
- load the current rerun `k = 3` clustered feature table from Notebook 08 outputs
- recompute PCA and Ward clustering using only **3** variables:
  - coastal/JPCZ signed-divergence ratio
  - Hokkaido `z850` anomaly
  - Sea of Japan `925 hPa` vorticity
- compare the new `3`-variable solution to the current `4`-variable solution
- report switching rates, PCA diagnostics, cluster counts, and cluster medians

Decision rule:
- if the `3`-variable solution stays close to the current `4`-variable solution, `T850` was probably not essential
- if the solution changes a lot, then a **masked-`T850` sensitivity** is the next step before changing the main Notebook 10 workflow

Scope note:
- this notebook does **not** rebuild climatology
- this notebook does **not** widen the `300 hPa` domain yet
- that more expensive upper-level work should wait until the subtype definition is settled


In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = "main"
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = False
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")

if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )
else:
    print("Using existing repo clone:", REPO_DIR)

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("Working directory:", os.getcwd())


In [ ]:
from itertools import permutations
from pathlib import Path
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from jpcz_catalog.analysis import add_japan_local_time_columns
from jpcz_catalog.subtypes import (
    assign_hierarchical_clusters,
    compute_mean_silhouette_score,
    evaluate_hierarchical_cluster_solutions,
    standardize_feature_table,
)

RUN_EXPORT_DIR_08 = Path("outputs/verification/objective_subtype_runs")
RUN_EXPORT_DIR_08.mkdir(parents=True, exist_ok=True)
SENSITIVITY_EXPORT_DIR = Path("outputs/verification/objective_subtype_t850_sensitivity")
SENSITIVITY_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = Path("outputs/verification/objective_subtype_t850_sensitivity_plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

CLUSTERED_K3_PATH = RUN_EXPORT_DIR_08 / "clustered_events_k3.csv"
PRIMARY_CLUSTER_COLUMN = "cluster_ward_3"
REDUCED_CLUSTER_COLUMN_RAW = "cluster_ward_3_no_t850_raw"
REDUCED_CLUSTER_COLUMN = "cluster_ward_3_no_t850"
SAVE_PLOTS = True
CLUSTER_COUNT_OPTIONS = [2, 3, 4]

FULL_FEATURE_COLUMNS = [
    "coastal_to_jpcz_mean_divergence_ratio",
    "hokkaido_min_z850_anomaly_tminus12_to_tplus12",
    "front_box_max_temp_gradient_850_tminus12_to_tplus12",
    "sea_of_japan_mean_vorticity_peak_925",
]
DROPPED_FEATURE = "front_box_max_temp_gradient_850_tminus12_to_tplus12"
REDUCED_FEATURE_COLUMNS = [column for column in FULL_FEATURE_COLUMNS if column != DROPPED_FEATURE]

FEATURE_LABELS = {
    "coastal_to_jpcz_mean_divergence_ratio": "Coastal/JPCZ signed-divergence ratio",
    "hokkaido_min_z850_anomaly_tminus12_to_tplus12": "Hokkaido minimum z850 anomaly",
    "front_box_max_temp_gradient_850_tminus12_to_tplus12": "Front-box maximum |grad T850|",
    "sea_of_japan_mean_vorticity_peak_925": "Sea of Japan mean 925 hPa vorticity",
}
FEATURE_UNITS = {
    "coastal_to_jpcz_mean_divergence_ratio": "unitless",
    "hokkaido_min_z850_anomaly_tminus12_to_tplus12": "gpm",
    "front_box_max_temp_gradient_850_tminus12_to_tplus12": "K (100 km)^-1",
    "sea_of_japan_mean_vorticity_peak_925": "1e-5 s^-1",
}


def maybe_copy_to_drive(path: Path, *, verbose: bool = True):
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return None
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if path.is_file():
        shutil.copy2(path, drive_path)
        if verbose:
            print("Copied to Drive:", drive_path)
        return drive_path
    return None


def restore_from_drive_cache(path: Path) -> bool:
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return False
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if not drive_path.exists():
        return False
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_path, path)
    print("Restored from Drive:", drive_path, "->", path)
    return True


def axis_label(column_name: str) -> str:
    label = FEATURE_LABELS.get(column_name, column_name)
    units = FEATURE_UNITS.get(column_name, "")
    if units and units != "unitless":
        return f"{label}\n[{units}]"
    return label


def best_cluster_label_mapping(reference_labels: pd.Series, candidate_labels: pd.Series):
    reference = reference_labels.astype(int)
    candidate = candidate_labels.astype(int).reindex(reference.index)
    unique_reference = sorted(reference.dropna().unique().tolist())
    unique_candidate = sorted(candidate.dropna().unique().tolist())
    if len(unique_reference) != len(unique_candidate):
        raise ValueError(
            "Expected the same number of clusters in the reference and candidate solutions, "
            f"got {unique_reference} versus {unique_candidate}."
        )

    best_mapping = None
    best_matches = -1
    for perm in permutations(unique_reference, len(unique_candidate)):
        mapping = dict(zip(unique_candidate, perm))
        mapped = candidate.map(mapping)
        match_count = int((mapped == reference).sum())
        if match_count > best_matches:
            best_matches = match_count
            best_mapping = mapping
    return best_mapping, best_matches


def compute_pca_diagnostics(feature_df: pd.DataFrame, feature_columns: list[str]):
    standardized_df, feature_means, feature_stds = standardize_feature_table(feature_df.copy(), columns=feature_columns)
    valid = standardized_df.dropna(axis=0, how="any")
    if valid.empty:
        raise ValueError("No complete rows are available for PCA.")

    matrix = valid.to_numpy(dtype=float)
    _, singular_values, vt = np.linalg.svd(matrix, full_matrices=False)
    n_components = min(3, matrix.shape[1])
    components = vt[:n_components]
    scores = matrix @ components.T

    total_variance = float((singular_values ** 2).sum())
    explained_variance_ratio = (singular_values[:n_components] ** 2) / total_variance

    score_df = pd.DataFrame(
        scores,
        index=valid.index,
        columns=[f"PC{i + 1}" for i in range(n_components)],
    )
    variance_df = pd.DataFrame(
        {
            "principal_component": [f"PC{i + 1}" for i in range(n_components)],
            "explained_variance_ratio": explained_variance_ratio,
            "explained_variance_percent": explained_variance_ratio * 100.0,
            "cumulative_explained_variance_ratio": np.cumsum(explained_variance_ratio),
            "cumulative_explained_variance_percent": np.cumsum(explained_variance_ratio) * 100.0,
        }
    )
    loadings_df = pd.DataFrame(
        components.T,
        index=feature_columns,
        columns=[f"PC{i + 1}" for i in range(n_components)],
    ).reset_index().rename(columns={"index": "feature_column"})
    loadings_df.insert(1, "plot_label", loadings_df["feature_column"].map(axis_label))
    loadings_df.insert(2, "units", loadings_df["feature_column"].map(lambda value: FEATURE_UNITS.get(value, "")))

    driver_rows = []
    for pc_name in [f"PC{i + 1}" for i in range(n_components)]:
        pc_subset = loadings_df[["feature_column", "plot_label", pc_name]].copy()
        pc_subset["abs_loading"] = pc_subset[pc_name].abs()
        pc_subset = pc_subset.sort_values("abs_loading", ascending=False).reset_index(drop=True)
        driver_rows.append(
            {
                "principal_component": pc_name,
                "explained_variance_percent": float(
                    variance_df.loc[variance_df["principal_component"] == pc_name, "explained_variance_percent"].iloc[0]
                ),
                "top_driver_feature": pc_subset.loc[0, "feature_column"],
                "top_driver_label": pc_subset.loc[0, "plot_label"],
                "top_driver_loading": float(pc_subset.loc[0, pc_name]),
                "second_driver_feature": pc_subset.loc[1, "feature_column"] if len(pc_subset) > 1 else "",
                "second_driver_label": pc_subset.loc[1, "plot_label"] if len(pc_subset) > 1 else "",
                "second_driver_loading": float(pc_subset.loc[1, pc_name]) if len(pc_subset) > 1 else np.nan,
            }
        )
    driver_df = pd.DataFrame(driver_rows)
    return standardized_df, score_df, variance_df, loadings_df, driver_df, feature_means, feature_stds


def cluster_count_table(cluster_labels: pd.Series) -> pd.DataFrame:
    counts = cluster_labels.dropna().astype(int).value_counts().sort_index().rename("n_events").to_frame().reset_index()
    counts.columns = ["cluster_id", "n_events"]
    counts["cluster_label"] = counts.apply(lambda row: f"Cluster {int(row['cluster_id'])}: n={int(row['n_events'])}", axis=1)
    return counts


In [ ]:
for path_to_restore in [CLUSTERED_K3_PATH]:
    if not path_to_restore.exists():
        restore_from_drive_cache(path_to_restore)

clustered_k3_df = pd.read_csv(CLUSTERED_K3_PATH)
clustered_k3_df = add_japan_local_time_columns(clustered_k3_df)
if PRIMARY_CLUSTER_COLUMN not in clustered_k3_df.columns:
    cluster_cols = [column for column in clustered_k3_df.columns if column.startswith("cluster_")]
    raise RuntimeError(f"Expected {PRIMARY_CLUSTER_COLUMN} in clustered_events_k3.csv, found {cluster_cols}")

current_cluster_labels = clustered_k3_df[PRIMARY_CLUSTER_COLUMN].astype(int)
current_cluster_counts_df = cluster_count_table(current_cluster_labels)

feature_scope_df = pd.DataFrame(
    {
        "feature_column": FULL_FEATURE_COLUMNS,
        "plot_label": [FEATURE_LABELS[column] for column in FULL_FEATURE_COLUMNS],
        "units": [FEATURE_UNITS[column] for column in FULL_FEATURE_COLUMNS],
        "used_in_current_4_feature_solution": True,
        "used_in_reduced_3_feature_solution": [column in REDUCED_FEATURE_COLUMNS for column in FULL_FEATURE_COLUMNS],
    }
)

print("Loaded current clustered k=3 rerun table from Notebook 08 outputs")
display(current_cluster_counts_df)
print("\nCurrent versus reduced feature scope")
display(feature_scope_df)


In [ ]:
baseline_standardized_df, baseline_scores_df, baseline_variance_df, baseline_loadings_df, baseline_driver_df, baseline_feature_means, baseline_feature_stds = compute_pca_diagnostics(
    clustered_k3_df,
    FULL_FEATURE_COLUMNS,
)
reduced_standardized_df, reduced_scores_df, reduced_variance_df, reduced_loadings_df, reduced_driver_df, reduced_feature_means, reduced_feature_stds = compute_pca_diagnostics(
    clustered_k3_df,
    REDUCED_FEATURE_COLUMNS,
)

reduced_quality_df = evaluate_hierarchical_cluster_solutions(
    reduced_standardized_df,
    cluster_counts=CLUSTER_COUNT_OPTIONS,
    method="ward",
)
reduced_cluster_labels_raw = assign_hierarchical_clusters(
    reduced_standardized_df,
    n_clusters=3,
    method="ward",
).rename(REDUCED_CLUSTER_COLUMN_RAW)
label_mapping, best_match_count = best_cluster_label_mapping(current_cluster_labels.loc[reduced_cluster_labels_raw.index], reduced_cluster_labels_raw)
reduced_cluster_labels = reduced_cluster_labels_raw.map(label_mapping).rename(REDUCED_CLUSTER_COLUMN)

clustered_k3_df[REDUCED_CLUSTER_COLUMN_RAW] = reduced_cluster_labels_raw.reindex(clustered_k3_df.index)
clustered_k3_df[REDUCED_CLUSTER_COLUMN] = reduced_cluster_labels.reindex(clustered_k3_df.index)

switch_mask = reduced_cluster_labels != current_cluster_labels.loc[reduced_cluster_labels.index]
switching_rows = [
    {
        "cluster_id": "all",
        "cluster_label": "All events",
        "n_events": int(len(reduced_cluster_labels)),
        "n_switched": int(switch_mask.sum()),
        "percent_switched": float(100.0 * switch_mask.mean()),
        "best_label_mapping": str(label_mapping),
        "best_label_match_count": int(best_match_count),
    }
]
for cluster_id in sorted(current_cluster_labels.dropna().astype(int).unique()):
    cluster_mask = current_cluster_labels.loc[reduced_cluster_labels.index] == int(cluster_id)
    switching_rows.append(
        {
            "cluster_id": int(cluster_id),
            "cluster_label": f"Cluster {int(cluster_id)}: n={int(cluster_mask.sum())}",
            "n_events": int(cluster_mask.sum()),
            "n_switched": int(switch_mask.loc[cluster_mask].sum()),
            "percent_switched": float(100.0 * switch_mask.loc[cluster_mask].mean()),
            "best_label_mapping": str(label_mapping),
            "best_label_match_count": int(best_match_count),
        }
    )
switching_df = pd.DataFrame(switching_rows)
switching_df["percent_switched"] = switching_df["percent_switched"].round(2)

crosstab_df = pd.crosstab(
    current_cluster_labels.rename("current_4_feature_cluster"),
    reduced_cluster_labels.rename("reduced_3_feature_cluster"),
)

reduced_cluster_counts_df = cluster_count_table(reduced_cluster_labels)
reduced_cluster_medians_df = (
    clustered_k3_df.loc[reduced_cluster_labels.index]
    .groupby(reduced_cluster_labels)[FULL_FEATURE_COLUMNS]
    .median()
    .round(3)
    .reset_index()
    .rename(columns={REDUCED_CLUSTER_COLUMN: "cluster_id"})
)

solution_summary_df = pd.DataFrame(
    [
        {
            "solution": "current_4_feature_k3",
            "n_features": len(FULL_FEATURE_COLUMNS),
            "features": ", ".join(FULL_FEATURE_COLUMNS),
            "silhouette": float(compute_mean_silhouette_score(baseline_standardized_df, current_cluster_labels)),
            "cluster_counts": ", ".join(
                [f"{int(row.cluster_id)}:{int(row.n_events)}" for row in current_cluster_counts_df.itertuples(index=False)]
            ),
        },
        {
            "solution": "reduced_3_feature_k3_no_t850",
            "n_features": len(REDUCED_FEATURE_COLUMNS),
            "features": ", ".join(REDUCED_FEATURE_COLUMNS),
            "silhouette": float(compute_mean_silhouette_score(reduced_standardized_df, reduced_cluster_labels)),
            "cluster_counts": ", ".join(
                [f"{int(row.cluster_id)}:{int(row.n_events)}" for row in reduced_cluster_counts_df.itertuples(index=False)]
            ),
        },
    ]
)
solution_summary_df["silhouette"] = solution_summary_df["silhouette"].round(5)

combined_variance_df = pd.concat(
    [
        baseline_variance_df.assign(solution="current_4_feature_k3"),
        reduced_variance_df.assign(solution="reduced_3_feature_k3_no_t850"),
    ],
    ignore_index=True,
)
combined_loadings_df = pd.concat(
    [
        baseline_loadings_df.assign(solution="current_4_feature_k3"),
        reduced_loadings_df.assign(solution="reduced_3_feature_k3_no_t850"),
    ],
    ignore_index=True,
)
combined_driver_df = pd.concat(
    [
        baseline_driver_df.assign(solution="current_4_feature_k3"),
        reduced_driver_df.assign(solution="reduced_3_feature_k3_no_t850"),
    ],
    ignore_index=True,
)

output_tables = {
    "k3_no_t850_solution_summary.csv": solution_summary_df,
    "k3_no_t850_switching_summary.csv": switching_df,
    "k3_no_t850_cluster_crosstab.csv": crosstab_df,
    "k3_no_t850_cluster_counts.csv": reduced_cluster_counts_df,
    "k3_no_t850_cluster_feature_medians.csv": reduced_cluster_medians_df,
    "k3_no_t850_quality_scan.csv": reduced_quality_df,
    "k3_pca_variance_comparison_no_t850.csv": combined_variance_df,
    "k3_pca_loadings_comparison_no_t850.csv": combined_loadings_df,
    "k3_pca_driver_comparison_no_t850.csv": combined_driver_df,
    "clustered_events_k3_with_no_t850_sensitivity.csv": clustered_k3_df,
}
for file_name, dataframe in output_tables.items():
    output_path = SENSITIVITY_EXPORT_DIR / file_name
    keep_index = file_name == "k3_no_t850_cluster_crosstab.csv"
    dataframe.to_csv(output_path, index=keep_index)
    maybe_copy_to_drive(output_path, verbose=False)

print("Current 4-feature versus reduced 3-feature k=3 comparison")
display(solution_summary_df)
print("\nReduced 3-feature quality scan across k = 2, 3, 4")
display(reduced_quality_df)
print("\nPercent of events that switch clusters after dropping T850 and best-matching the labels")
display(switching_df)
print("\nCurrent 4-feature versus reduced 3-feature cluster cross-tab")
display(crosstab_df)
print("\nReduced 3-feature cluster medians, including the dropped T850 variable for comparison")
display(reduced_cluster_medians_df)
print("\nPCA variance comparison")
display(combined_variance_df)
print("\nTop PCA loading drivers")
display(combined_driver_df)


In [ ]:
cluster_colors = {1: "#1f77b4", 2: "#ff7f0e", 3: "#2ca02c"}

fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
plot_specs = [
    (
        axes[0],
        baseline_scores_df,
        current_cluster_labels.loc[baseline_scores_df.index].astype(int),
        baseline_variance_df,
        "Current 4-feature PCA space",
    ),
    (
        axes[1],
        reduced_scores_df,
        reduced_cluster_labels.loc[reduced_scores_df.index].astype(int),
        reduced_variance_df,
        "Reduced 3-feature PCA space (no T850)",
    ),
]

for ax, score_df, labels, variance_df, title in plot_specs:
    for cluster_id in sorted(labels.unique()):
        mask = labels == cluster_id
        ax.scatter(
            score_df.loc[mask, "PC1"],
            score_df.loc[mask, "PC2"],
            s=28,
            alpha=0.8,
            color=cluster_colors.get(int(cluster_id), "gray"),
            label=f"Cluster {int(cluster_id)}",
        )
        centroid_x = float(score_df.loc[mask, "PC1"].mean())
        centroid_y = float(score_df.loc[mask, "PC2"].mean())
        ax.text(centroid_x, centroid_y, f"C{int(cluster_id)}", fontsize=10, weight="bold")
    pc1_pct = float(variance_df.loc[variance_df["principal_component"] == "PC1", "explained_variance_percent"].iloc[0])
    pc2_pct = float(variance_df.loc[variance_df["principal_component"] == "PC2", "explained_variance_percent"].iloc[0])
    ax.set_xlabel(f"PC1 ({pc1_pct:.1f}% variance)")
    ax.set_ylabel(f"PC2 ({pc2_pct:.1f}% variance)")
    ax.set_title(title)
    ax.grid(alpha=0.25)
    ax.legend(loc="best")

fig.suptitle(
    "PCA comparison: current 4-feature solution versus reduced 3-feature solution without T850",
    fontsize=13,
)
scatter_path = PLOT_DIR / "k3_pca_scatter_full_vs_reduced_no_t850.png"
if SAVE_PLOTS:
    fig.savefig(scatter_path, dpi=180, bbox_inches="tight")
    maybe_copy_to_drive(scatter_path, verbose=False)
plt.show()

variance_fig, variance_ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
variance_compare_plot_df = combined_variance_df.copy()
variance_pivot = variance_compare_plot_df.pivot(index="principal_component", columns="solution", values="explained_variance_percent")
variance_pivot.plot(kind="bar", ax=variance_ax)
variance_ax.set_ylabel("Explained variance [%]")
variance_ax.set_title("Explained variance comparison after dropping T850")
variance_ax.grid(axis="y", alpha=0.25)
variance_path = PLOT_DIR / "k3_pca_variance_full_vs_reduced_no_t850.png"
if SAVE_PLOTS:
    variance_fig.savefig(variance_path, dpi=180, bbox_inches="tight")
    maybe_copy_to_drive(variance_path, verbose=False)
plt.show()

print("Saved PCA sensitivity plots")
display(pd.DataFrame({
    "plot": ["PCA scatter", "PCA variance comparison"],
    "path": [str(scatter_path), str(variance_path)],
}))


## How to read the result

- If the switching rate stays very low and the `k = 3` silhouette stays similar, then `T850` was probably **not essential** to the subtype definition.
- If the `Cluster 2` versus `Cluster 3` boundary changes a lot, then `T850` was doing real work in the current subtype split.
- If the reduced `3`-feature solution looks unstable or much less interpretable, the next step is **not** to drop `T850` outright. The next step is a **masked-`T850` sensitivity** where high terrain is excluded and the feature is rebuilt more carefully.

Recommended next decision after running this notebook:
1. If the solution barely changes: consider dropping `T850` from the subtype-definition feature set.
2. If the solution changes materially: keep `T850` in play for now, then test a high-terrain mask before changing Notebook 10.
3. Only after that subtype-definition decision is settled, return to the wider-domain `300 hPa` composite work.
